# Comparer la robustesse avec et sans augmentation — v2 (CBAM résiduel, conditions hors plage)

**Objectif :** vérifier si l’augmentation de luminosité et de contraste utilisée pendant l’apprentissage (notebook 09 v2) améliore les performances sur les images perturbées, et si le CBAM résiduel garde ou retrouve son avantage sur le CNN classique.

Le notebook recharge **douze modèles**, sans les réentraîner :
- les six modèles **sans augmentation** évalués dans le notebook 08 v2 (dossier `robustesse_validation_v2`) ;
- les six modèles **avec augmentation** du notebook 09 v2 (dossier `augmentation_luminosite_contraste_v2`).

Chaque architecture garde sa propre configuration retenue : classique (LR 0,001 ; Dropout 0,5), CBAM résiduel (LR 0,0005 ; Dropout 0,3).

## Conditions évaluées

| Groupe | Conditions | Statut |
|---|---|---|
| Reprises du notebook 08 v2 | Originale ; luminosité × 0,90 et × 1,10 ; contraste × 0,90 et × 1,10 ; bruit σ = 0,01 et 0,03 | Identiques au notebook 08 v2. Les résultats sans augmentation sont **réutilisés** tels quels. |
| **Hors plage d’augmentation** (nouvelles) | Luminosité × 0,80 et × 1,20 ; contraste × 0,80 et × 1,20 | **Fixées le 24/09/2026, avant de connaître les résultats du notebook 09 v2.** Évaluées ici pour les douze modèles. |

**Pourquoi des conditions hors plage ?** L’augmentation tire des facteurs entre 0,90 et 1,10, soit exactement les niveaux de luminosité et de contraste du notebook 08. Une amélioration sur ces niveaux montrerait seulement que le modèle tolère des variations vues pendant l’entraînement. Les conditions à 0,80 et 1,20 testent si ce gain **s’étend au-delà** de la plage apprise. Le bruit gaussien n’a jamais été ajouté pendant l’entraînement : il joue aussi le rôle de condition non vue.

Nombre de passages de validation : 6 modèles sans augmentation × 5 conditions (l’originale, pour contrôler le rechargement, et les 4 nouvelles) = 30, plus 6 modèles avec augmentation × 11 conditions = 66, soit **96 passages**. Les passages terminés sont sauvegardés et réutilisés après une interruption.

**Trois mesures complémentaires :**

- **Gain de score** = score avec augmentation − score sans augmentation, pour la même architecture, la même graine et la même condition. Positif : meilleure performance avec augmentation.
- **Gain de variation** = (perturbé − original) avec augmentation − (perturbé − original) sans augmentation. Positif : dégradation moins forte par rapport au score original propre à chaque modèle.
- **Écart entre architectures** = F1 macro du CBAM − F1 macro du classique, par graine et par condition, avec et sans augmentation. C’est la mesure qui répond à la question du mémoire.

Il faut lire ces mesures ensemble : un modèle moins performant sur l’original peut perdre moins de points sans mieux classer les images perturbées.

**Limites :** l’augmentation a été choisie après observation de ces perturbations sur validation. Cette comparaison reste exploratoire, sur une validation déjà utilisée pour sélectionner les modèles et les époques. Trois graines donnent peu d’information sur l’incertitude. La graine 42 a servi à sélectionner les hyperparamètres. Le test avait déjà été consulté dans le projet ; il n’est pas chargé ici. Les niveaux 0,80 et 1,20 sont des choix expérimentaux, pas des normes cliniques. Ces résultats ne démontrent ni une généralisation à d’autres centres ni une fiabilité clinique.

**À faire :** exécuter les cellules dans l’ordre avec le noyau `memoire_cnn`. Ne pas lancer deux copies simultanément.


In [ ]:
from pathlib import Path, PureWindowsPath
import hashlib
import json
import sys
import time
import numpy as np
import pandas as pd
import PIL
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn import __version__ as version_sklearn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display

# Si le projet change de dossier, modifier uniquement PROJET.
PROJET = Path(r"C:\Users\David KUNYIMA\OneDrive\Desktop\Memoire_CNN_Attention")
IMAGES = PROJET / "archive" / "chest_xray"
REPARTITION = PROJET / "repartitions" / "repartition_20260917_102959_713272"
FICHIER_RESULTATS = PROJET / "optimisation" / "augmentation_luminosite_contraste_v2" / "resultats_augmentation.csv"
DOSSIER_REFERENCE = PROJET / "optimisation" / "robustesse_validation_v2"
FICHIER_REFERENCE = DOSSIER_REFERENCE / "resultats_robustesse.csv"
SORTIE = PROJET / "optimisation" / "robustesse_augmentation_v2"
APPAREIL = torch.device("cpu")
LOT, TAILLE = 16, 224
GRAINE_BRUIT = 314159
CLASSES = {"NORMAL": 0, "PNEUMONIA": 1}
CONDITIONS_REFERENCE = [
    {"id": "originale", "type": "identite", "valeur": 0.0, "libelle": "Originale"},
    {"id": "luminosite_090", "type": "luminosite", "valeur": 0.9, "libelle": "Luminosité × 0,90"},
    {"id": "luminosite_110", "type": "luminosite", "valeur": 1.1, "libelle": "Luminosité × 1,10"},
    {"id": "contraste_090", "type": "contraste", "valeur": 0.9, "libelle": "Contraste × 0,90"},
    {"id": "contraste_110", "type": "contraste", "valeur": 1.1, "libelle": "Contraste × 1,10"},
    {"id": "bruit_001", "type": "bruit", "valeur": 0.01, "libelle": "Bruit σ = 0,01"},
    {"id": "bruit_003", "type": "bruit", "valeur": 0.03, "libelle": "Bruit σ = 0,03"},
]
# Conditions hors de la plage d'augmentation [0,90 ; 1,10], fixées le 24/09/2026,
# AVANT de connaître les résultats du notebook 09 v2. Ne pas les modifier après avoir vu les résultats.
CONDITIONS_HORS_PLAGE = [
    {"id": "luminosite_080", "type": "luminosite", "valeur": 0.8, "libelle": "Luminosité × 0,80"},
    {"id": "luminosite_120", "type": "luminosite", "valeur": 1.2, "libelle": "Luminosité × 1,20"},
    {"id": "contraste_080", "type": "contraste", "valeur": 0.8, "libelle": "Contraste × 0,80"},
    {"id": "contraste_120", "type": "contraste", "valeur": 1.2, "libelle": "Contraste × 1,20"},
]
CONDITIONS = CONDITIONS_REFERENCE + CONDITIONS_HORS_PLAGE
IDS_HORS_PLAGE = [c["id"] for c in CONDITIONS_HORS_PLAGE]
exiger_ids = [c["id"] for c in CONDITIONS]
assert len(set(exiger_ids)) == len(exiger_ids) and exiger_ids[0] == "originale"
# Configuration retenue propre à chaque architecture (notebooks 07 v2, 08 v2 et 09 v2).
CONFIGURATIONS = {
    "CNN_classique": {"indice": 2, "lr": 0.001, "dropout": 0.5},
    "CNN_CBAM": {"indice": 3, "lr": 0.0005, "dropout": 0.3},
}
torch.use_deterministic_algorithms(True)

def exiger(condition, message):
    if not condition:
        raise RuntimeError(message)

def empreinte_fichier(chemin):
    h = hashlib.sha256()
    with open(chemin, "rb") as fichier:
        for bloc in iter(lambda: fichier.read(1024 * 1024), b""):
            h.update(bloc)
    return h.hexdigest()

def ecrire_json(obj, chemin):
    temporaire = chemin.with_suffix(".tmp")
    temporaire.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    temporaire.replace(chemin)

def ecrire_csv(tableau, chemin, **options):
    temporaire = chemin.with_suffix(".tmp")
    tableau.to_csv(temporaire, index=False, encoding="utf-8-sig", float_format="%.17g", **options)
    temporaire.replace(chemin)

print("Matériel :", APPAREIL)
tableau_conditions = pd.DataFrame(CONDITIONS)[["libelle", "type", "valeur"]]
tableau_conditions["groupe"] = ["hors plage (nouvelle)" if c["id"] in IDS_HORS_PLAGE else "reprise du notebook 08 v2" for c in CONDITIONS]
display(tableau_conditions)


## 1. Préparer les mêmes images et les perturbations

Les images préparées sont gardées en mémoire pour éviter de rouvrir 96 fois les mêmes fichiers. Les images et leurs masques occupent environ 96 Mo. Les fichiers d’origine restent inchangés.

Le contraste est centré sur la moyenne des pixels du rectangle de l’image, hors padding ajouté. Le bruit dépend de l’identifiant de l’image, et non de l’ordre des lots ou de la graine du modèle.


In [ ]:
def preparer_avec_masque(image):
    gris = image.convert("L")
    preparee = ImageOps.pad(
        gris, (TAILLE, TAILLE), method=Image.Resampling.BILINEAR,
        color=0, centering=(0.5, 0.5)
    )
    # Toute l'image source est marquée, même ses pixels initialement noirs.
    blanc = Image.new("L", gris.size, color=255)
    masque = ImageOps.pad(
        blanc, (TAILLE, TAILLE), method=Image.Resampling.NEAREST,
        color=0, centering=(0.5, 0.5)
    )
    return np.array(preparee, dtype=np.uint8), np.array(masque) > 0

def bruit_image(identifiant, forme):
    cle = f"{GRAINE_BRUIT}|{identifiant}".encode("utf-8")
    graine = int.from_bytes(hashlib.sha256(cle).digest()[:8], "little")
    generateur = np.random.Generator(np.random.PCG64(graine))
    return generateur.standard_normal(forme).astype(np.float32)

def transformer(tableau_u8, masque, condition, identifiant):
    x = tableau_u8.astype(np.float32) / 255.0
    nature, valeur = condition["type"], condition["valeur"]
    if nature == "identite":
        return x
    resultat = x.copy()
    if nature == "luminosite":
        resultat[masque] = x[masque] * np.float32(valeur)
    elif nature == "contraste":
        moyenne = x[masque].mean(dtype=np.float64)
        resultat[masque] = (x[masque] - moyenne) * valeur + moyenne
    elif nature == "bruit":
        bruit = bruit_image(identifiant, x.shape)
        resultat[masque] = x[masque] + np.float32(valeur) * bruit[masque]
    else:
        raise ValueError(f"Transformation inconnue : {nature}")
    return np.clip(resultat, 0.0, 1.0).astype(np.float32)

exiger(FICHIER_RESULTATS.is_file(), f"Fichier absent : {FICHIER_RESULTATS}")
val_df = pd.read_csv(REPARTITION / "val.csv", dtype={"groupe": str, "empreinte": str})
colonnes = {"chemin_relatif", "groupe", "empreinte", "classe"}
exiger(colonnes.issubset(val_df.columns), "Colonnes de validation manquantes.")
exiger(not val_df[list(colonnes)].isna().any().any(), "Données de validation incomplètes.")
exiger(val_df["chemin_relatif"].is_unique, "Chemins de validation dupliqués.")
exiger(val_df["classe"].value_counts().to_dict() == {"NORMAL": 269, "PNEUMONIA": 684}, "Effectifs de validation inattendus.")
REELS = val_df["classe"].map(CLASSES).to_numpy(dtype=np.int64)
IDENTIFIANTS = val_df["chemin_relatif"].tolist()
PREPAREES = np.empty((len(val_df), TAILLE, TAILLE), dtype=np.uint8)
MASQUES = np.empty_like(PREPAREES, dtype=bool)
empreinte_images = hashlib.sha256()
for i, identifiant in enumerate(tqdm(IDENTIFIANTS, desc="Préparation de la validation")):
    with Image.open(IMAGES / identifiant) as image:
        PREPAREES[i], MASQUES[i] = preparer_avec_masque(image)
    exiger(MASQUES[i].any(), "Une image ne possède aucun pixel utilisable.")
    exiger((PREPAREES[i][~MASQUES[i]] == 0).all(), "Erreur de géométrie du padding.")
    empreinte_images.update(identifiant.encode("utf-8") + b"\0")
    empreinte_images.update(PREPAREES[i].tobytes())
    empreinte_images.update(MASQUES[i].tobytes())
SHA_IMAGES = empreinte_images.hexdigest()
print("Validation :", len(val_df), "images. Mémoire images + masques :", round((PREPAREES.nbytes + MASQUES.nbytes) / 1e6, 1), "Mo")


## 2. Vérifier visuellement les transformations

Deux exemples fixes sont affichés : le premier NORMAL et le premier PNEUMONIA dans la liste de validation. Ils ne sont pas choisis en fonction des prédictions. Les étiquettes viennent du dataset.


In [ ]:
fig, axes = plt.subplots(2, len(CONDITIONS), figsize=(2.5 * len(CONDITIONS), 6))
for ligne, classe in enumerate(CLASSES):
    i = int(np.flatnonzero(val_df["classe"].to_numpy() == classe)[0])
    for colonne, condition in enumerate(CONDITIONS):
        x = transformer(PREPAREES[i], MASQUES[i], condition, IDENTIFIANTS[i])
        exiger(x.dtype == np.float32 and np.isfinite(x).all(), "Valeurs de pixels incorrectes.")
        exiger(0 <= x.min() <= x.max() <= 1, "Pixels hors de [0, 1].")
        exiger((x[~MASQUES[i]] == 0).all(), "Le padding a été modifié.")
        axes[ligne, colonne].imshow(x, cmap="gray", vmin=0, vmax=1)
        axes[ligne, colonne].set_title(condition["libelle"], fontsize=10)
        axes[ligne, colonne].axis("off")
    axes[ligne, 0].set_title(f"{classe}\nOriginale", fontsize=10)
fig.suptitle("Exemples de validation — transformations appliquées séparément", fontsize=14)
fig.tight_layout()
plt.show()
print("Les bordures ajoutées sont conservées. Une copie de cette figure sera enregistrée après les contrôles du protocole.")
figure_exemples = fig


## 3. Vérifier la référence et recharger les douze modèles

Les fichiers du notebook 08 v2 restent inchangés. La vérification de leurs métadonnées empêche de comparer des résultats obtenus sur des images, des conditions ou des modèles différents.

Les six modèles **sans augmentation** sont rechargés depuis les sauvegardes enregistrées dans le protocole du notebook 08 v2 ; leur empreinte SHA-256 doit être identique. Ils ne servent qu’aux quatre nouvelles conditions (plus l’originale, pour contrôler qu’ils redonnent exactement les scores de référence).

Chaque modèle **avec augmentation** doit être relié à la bonne expérience sans augmentation. Le chargement met tous les modèles en mode évaluation et désactive le calcul des gradients.


In [ ]:
class CNNClassique(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.bloc1 = nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.bloc2 = nn.Sequential(nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.bloc3 = nn.Sequential(nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.reduction = nn.AdaptiveAvgPool2d((4,4))
        self.classificateur = nn.Sequential(nn.Flatten(),nn.Linear(64*4*4,64),nn.ReLU(),nn.Dropout(dropout),nn.Linear(64,2))
    def forward(self,x):
        x=self.bloc3(self.bloc2(self.bloc1(x)))
        return self.classificateur(self.reduction(x))

class AttentionCanaux(nn.Module):
    def __init__(self, canaux, reduction=16):
        super().__init__()
        dim=max(1,canaux//reduction)
        self.reseau_partage=nn.Sequential(
            nn.Conv2d(canaux,dim,1,bias=False),nn.ReLU(),
            nn.Conv2d(dim,canaux,1,bias=False))
    def forward(self,x):
        moyenne=x.mean(dim=(2,3),keepdim=True)
        maximum=x.amax(dim=(2,3),keepdim=True)
        return x*torch.sigmoid(self.reseau_partage(moyenne)+self.reseau_partage(maximum))

class AttentionSpatiale(nn.Module):
    def __init__(self):
        super().__init__()
        self.convolution=nn.Conv2d(2,1,7,padding=3,bias=False)
    def forward(self,x):
        resume=torch.cat([x.mean(dim=1,keepdim=True),x.amax(dim=1,keepdim=True)],dim=1)
        return x*torch.sigmoid(self.convolution(resume))

class CBAM(nn.Module):
    def __init__(self,canaux=64,reduction=16):
        super().__init__()
        self.attention_canaux=AttentionCanaux(canaux,reduction)
        self.attention_spatiale=AttentionSpatiale()
    def forward(self,x):
        # v2 : connexion résiduelle, identique aux notebooks 06 v2 à 09 v2.
        attenue = self.attention_spatiale(self.attention_canaux(x))
        return x + attenue

class CNNAvecCBAM(CNNClassique):
    def __init__(self,dropout=0.3):
        super().__init__(dropout)
        self.cbam=CBAM(64,16)
    def forward(self,x):
        x=self.bloc3(self.bloc2(self.bloc1(x)))
        return self.classificateur(self.reduction(self.cbam(x)))


In [ ]:
ARCHITECTURES = {
    "CNN_classique": (CNNClassique, "CNNClassique_89026_v1", 89026),
    "CNN_CBAM": (CNNAvecCBAM, "CNNAvecCBAM_89636_apres_bloc3_r16_k7_residuel_v2", 89636),
}
VERSIONS = {
    "python": sys.version, "torch": str(torch.__version__), "numpy": np.__version__,
    "pandas": pd.__version__, "pillow": PIL.__version__, "sklearn": version_sklearn,
}
PRETRAITEMENT_BASE = "L; ImageOps.pad 224 bilineaire noir centre; /255"
PRETRAITEMENT = PRETRAITEMENT_BASE + "; aucune augmentation"
METRIQUES_COMPARAISON = ["f1_macro", "exactitude", "precision_pneumonie", "rappel_pneumonie", "specificite_NORMAL", "f1_pneumonie", "auc_roc"]
AUGMENTATION_ATTENDUE = {'version': 'luminosite_contraste_v1', 'luminosite': [0.9, 1.1], 'contraste': [0.9, 1.1], 'probabilite': 1.0, 'ordre': 'aleatoire entre les deux ordres possibles', 'zone': 'rectangle image redimensionnee, padding ajoute preserve', 'contraste_centre': 'moyenne float64 du rectangle au moment de l’operation', 'ecretage': '[0,1] apres chaque operation; pixels float32', 'generateur': 'numpy PCG64, SHA256(version|graine|epoque|chemin), 8 premiers octets little-endian'}
COUPLES = {(m, g) for m in ARCHITECTURES for g in [42, 123, 2026]}
CLES_RESULTATS = ["modele", "graine", "condition"]
SHA_VAL = empreinte_fichier(REPARTITION / "val.csv")

def verifier_bilan(tableau, conditions):
    colonnes = set(CLES_RESULTATS + ["epoque", "libelle", "n_images", "vrais_negatifs", "faux_positifs", "faux_negatifs", "vrais_positifs"] + METRIQUES_COMPARAISON)
    colonnes |= {f"original_{m}" for m in METRIQUES_COMPARAISON} | {f"variation_{m}" for m in METRIQUES_COMPARAISON}
    exiger(colonnes.issubset(tableau.columns), "Bilan incomplet : colonnes manquantes.")
    attendu_n = len(COUPLES) * len(conditions)
    exiger(len(tableau) == attendu_n and not tableau.duplicated(CLES_RESULTATS).any(), f"Il faut {attendu_n} évaluations distinctes.")
    attendus = {(m, g, c["id"]) for m, g in COUPLES for c in conditions}
    exiger(set(map(tuple, tableau[CLES_RESULTATS].to_numpy())) == attendus, "Modèles, graines ou conditions inattendus.")
    exiger(not tableau[list(colonnes)].isna().any().any(), "Valeurs manquantes dans le bilan.")
    exiger((tableau["n_images"] == len(val_df)).all(), "Effectif de validation différent.")
    comptes = tableau[["vrais_negatifs", "faux_positifs", "faux_negatifs", "vrais_positifs"]].to_numpy(dtype=float)
    exiger(np.isfinite(comptes).all() and (comptes >= 0).all() and np.array_equal(comptes, np.floor(comptes)), "Comptes de confusion invalides.")
    exiger(((tableau["vrais_negatifs"] + tableau["faux_positifs"]) == 269).all(), "Effectif NORMAL différent.")
    exiger(((tableau["vrais_positifs"] + tableau["faux_negatifs"]) == 684).all(), "Effectif PNEUMONIA différent.")
    exiger(np.allclose(tableau["exactitude"], (tableau["vrais_negatifs"] + tableau["vrais_positifs"])/len(val_df), rtol=0, atol=1e-10), "Exactitudes incohérentes.")
    tn, fp, fn, tp = comptes.T
    f1_p = 2*tp/(2*tp+fp+fn)
    f1_n = 2*tn/(2*tn+fp+fn)
    calculs = {"precision_pneumonie": np.divide(tp, tp+fp, out=np.zeros_like(tp), where=(tp+fp)>0),
               "rappel_pneumonie": tp/(tp+fn), "specificite_NORMAL": tn/(tn+fp),
               "f1_pneumonie": f1_p, "f1_macro": (f1_p+f1_n)/2}
    for mesure, calcule in calculs.items():
        exiger(np.allclose(tableau[mesure], calcule, rtol=0, atol=1e-10), f"Mesure incohérente avec la matrice : {mesure}.")
    originals = tableau[tableau["condition"] == "originale"].set_index(["modele", "graine"])
    for m in METRIQUES_COMPARAISON:
        exiger(np.isfinite(tableau[[m, f"original_{m}", f"variation_{m}"]].to_numpy(dtype=float)).all(), "Mesures non finies.")
        exiger(tableau[m].between(0, 1).all(), f"Mesure hors de [0, 1] : {m}.")
        exiger(np.allclose(tableau[f"variation_{m}"], tableau[m]-tableau[f"original_{m}"], rtol=0, atol=1e-10), f"Variation incorrecte : {m}.")
        for ligne in tableau.to_dict("records"):
            attendu = originals.loc[(ligne["modele"], ligne["graine"]), m]
            exiger(abs(ligne[f"original_{m}"]-attendu)<1e-10, "Référence originale incohérente.")
    return tableau

# ---------- Référence sans augmentation (notebook 08 v2) ----------
exiger(FICHIER_REFERENCE.is_file(), f"Bilan de référence absent : {FICHIER_REFERENCE}")
fichier_protocole_reference = DOSSIER_REFERENCE / "protocole.json"
exiger(fichier_protocole_reference.is_file(), f"Protocole de référence absent : {fichier_protocole_reference}")
PROTOCOLE_REFERENCE = json.loads(fichier_protocole_reference.read_text(encoding="utf-8"))
attendus_reference = {
    "version_analyse": "robustesse_validation_v2", "ensemble": "val", "effectif": len(val_df),
    "conditions": CONDITIONS_REFERENCE, "lot": LOT, "appareil": str(APPAREIL), "classes": CLASSES,
    "pretraitement": PRETRAITEMENT, "sha256_val_csv": SHA_VAL,
    "sha256_images_preparees_et_masques": SHA_IMAGES,
    "versions": VERSIONS, "threads_torch": torch.get_num_threads(),
    "bruit": {"graine": GRAINE_BRUIT, "generateur": "PCG64", "cle": "SHA256(graine|chemin_relatif), 8 premiers octets little-endian", "realisations_par_image": 1},
    "zone_perturbee": "rectangle image redimensionnee; padding ajoute preserve",
    "contraste": "moyenne sur le rectangle image; float64 pour le calcul de cette moyenne",
    "ecretage": "[0,1] apres transformation; retour float32",
    "prediction": "argmax des 2 logits, NORMAL en cas d’egalite",
}
for cle, valeur in attendus_reference.items():
    exiger(PROTOCOLE_REFERENCE.get(cle) == valeur,
           f"La référence de robustesse diffère pour {cle}. Reprendre les mêmes données, réglages et environnement que le notebook 08 v2.")
RESULTATS_SANS_REF = verifier_bilan(pd.read_csv(FICHIER_REFERENCE), CONDITIONS_REFERENCE)
exiger(len(PROTOCOLE_REFERENCE["modeles"]) == 6, "Références de modèles incomplètes.")
REF_MODELES = {(r["modele"], r["graine"]): r for r in PROTOCOLE_REFERENCE["modeles"]}
exiger(set(REF_MODELES) == COUPLES, "Modèles de référence incompatibles.")
for (nom, graine), reference in REF_MODELES.items():
    selection = RESULTATS_SANS_REF[(RESULTATS_SANS_REF["modele"] == nom) & (RESULTATS_SANS_REF["graine"] == graine)]
    exiger((selection["epoque"] == reference["epoque"]).all(), "Époque de référence incohérente.")
    exiger(reference.get("lr") == CONFIGURATIONS[nom]["lr"] and reference.get("dropout") == CONFIGURATIONS[nom]["dropout"],
           f"La référence {nom}, graine {graine} n’utilise pas la configuration retenue.")

def retrouver_sauvegarde(texte):
    morceaux = PureWindowsPath(str(texte)).parts
    positions = [i for i, nom in enumerate(morceaux) if nom == "optimisation"]
    exiger(len(positions) == 1, f"Chemin de sauvegarde inattendu : {texte}")
    relatifs = morceaux[positions[0]:]
    exiger(".." not in relatifs, "Chemin relatif incorrect.")
    return PROJET.joinpath(*relatifs)

def charger_modele(nom, graine, chemin, pretraitement_attendu):
    classe, architecture, nombre_parametres = ARCHITECTURES[nom]
    exiger(chemin.is_file(), f"Sauvegarde introuvable : {chemin}")
    etat = torch.load(chemin, map_location="cpu", weights_only=True)
    protocole = etat["protocole"]
    exiger(protocole["architecture"] == architecture, f"Architecture différente : {nom}.")
    exiger(protocole["graine"] == graine, f"Graine incompatible : {nom}, {graine}.")
    exiger(protocole["sha256_csv"]["val"] == SHA_VAL, "Répartition de validation différente.")
    exiger(protocole["classes"] == CLASSES and protocole["pretraitement"] == pretraitement_attendu, "Classes ou préparation incompatibles.")
    exiger(etat["configuration"] == CONFIGURATIONS[nom], "Sauvegarde d’une autre configuration.")
    meilleur = max(etat["historique"], key=lambda r: r["val_f1_macro"])
    exiger(meilleur["epoque"] == etat["epoque"], "La sauvegarde n’est pas celle de la meilleure époque.")
    with torch.random.fork_rng(devices=[]):
        modele = classe(dropout=CONFIGURATIONS[nom]["dropout"]).to(APPAREIL)
        modele.load_state_dict(etat["modele"], strict=True)
    exiger(sum(p.numel() for p in modele.parameters()) == nombre_parametres, "Nombre de paramètres inattendu.")
    modele.eval()
    modele.requires_grad_(False)
    return modele, etat, protocole, meilleur

MODELES, INFORMATIONS = {}, []

# ---------- Six modèles sans augmentation (mêmes fichiers que le notebook 08 v2) ----------
for (nom, graine), reference in sorted(REF_MODELES.items()):
    chemin = PROJET.joinpath(*reference["sauvegarde_relative"].split("/"))
    exiger(empreinte_fichier(chemin) == reference["sha256_sauvegarde"], f"La sauvegarde {chemin} a changé depuis le notebook 08 v2.")
    modele, etat, protocole, meilleur = charger_modele(nom, graine, chemin, PRETRAITEMENT)
    exiger(int(etat["epoque"]) == int(reference["epoque"]), "Époque de référence incohérente.")
    MODELES[("sans", nom, graine)] = modele
    INFORMATIONS.append({
        "variante": "sans", "modele": nom, "graine": graine, "epoque": int(etat["epoque"]),
        "architecture": ARCHITECTURES[nom][1], "sha256_sauvegarde": reference["sha256_sauvegarde"],
        "sauvegarde_relative": reference["sauvegarde_relative"],
    })
    del etat

# ---------- Six modèles avec augmentation (notebook 09 v2) ----------
table_modeles = pd.read_csv(FICHIER_RESULTATS)
necessaires = {"modele", "graine", "epoque_retenue", "lr", "dropout", "augmentation", "sauvegarde", "val_f1_macro", "val_exactitude", "val_rappel_pneumonie", "val_specificite_NORMAL"}
exiger(necessaires.issubset(table_modeles.columns), "Le tableau des nouveaux modèles est incomplet.")
exiger(not table_modeles[list(necessaires)].isna().any().any(), "Information manquante pour les nouveaux modèles.")
exiger(len(table_modeles)==6 and not table_modeles.duplicated(["modele","graine"]).any(), "Il faut six nouveaux modèles distincts.")
exiger(set(zip(table_modeles["modele"], table_modeles["graine"])) == COUPLES, "Couples modèle/graine inattendus.")
exiger((table_modeles["augmentation"] == AUGMENTATION_ATTENDUE["version"]).all(), "Autre protocole d’augmentation.")
for nom_config, reglages in CONFIGURATIONS.items():
    sous_table = table_modeles[table_modeles["modele"] == nom_config]
    exiger(np.allclose(sous_table["lr"], reglages["lr"], rtol=0, atol=1e-12)
           and np.allclose(sous_table["dropout"], reglages["dropout"], rtol=0, atol=1e-12),
           f"Réglages différents de ceux retenus pour {nom_config}.")
table_modeles = table_modeles.sort_values(["modele", "graine"]).reset_index(drop=True)

for ligne in table_modeles.to_dict("records"):
    nom, graine = ligne["modele"], int(ligne["graine"])
    chemin = retrouver_sauvegarde(ligne["sauvegarde"])
    modele, etat, protocole, meilleur = charger_modele(nom, graine, chemin, PRETRAITEMENT_BASE)
    exiger(int(etat["epoque"]) == int(ligne["epoque_retenue"]), "Époque incompatible.")
    exiger(protocole["augmentation_train"] == AUGMENTATION_ATTENDUE and protocole["augmentation_validation"] is False, "Protocole d’augmentation incompatible.")
    reference = REF_MODELES[(nom, graine)]
    exiger(protocole["origine_essai"]["sha256_reference"] == reference["sha256_sauvegarde"], "Le nouveau modèle n’est pas lié à la bonne référence sans augmentation.")
    for metrique in ["f1_macro", "exactitude", "rappel_pneumonie", "specificite_NORMAL"]:
        exiger(abs(float(meilleur[f"val_{metrique}"])-float(ligne[f"val_{metrique}"]))<1e-10, "Le tableau et la sauvegarde ne correspondent pas.")
    MODELES[("avec", nom, graine)] = modele
    INFORMATIONS.append({
        "variante": "avec", "modele": nom, "graine": graine, "epoque": int(etat["epoque"]),
        "architecture": ARCHITECTURES[nom][1], "sha256_sauvegarde": empreinte_fichier(chemin),
        "sauvegarde_relative": chemin.relative_to(PROJET).as_posix(),
        "sha256_reference_sans_augmentation": reference["sha256_sauvegarde"],
    })
    del etat

PROTOCOLE = {
    **attendus_reference, "version_analyse": "comparaison_robustesse_augmentation_v2",
    "conditions": CONDITIONS, "conditions_reference_08": CONDITIONS_REFERENCE,
    "conditions_hors_plage": CONDITIONS_HORS_PLAGE,
    "fixation_conditions_hors_plage": "2026-09-24, avant les resultats du notebook 09 v2",
    "configurations": CONFIGURATIONS,
    "sha256_resultats_augmentation": empreinte_fichier(FICHIER_RESULTATS),
    "sha256_bilan_sans_augmentation": empreinte_fichier(FICHIER_REFERENCE),
    "sha256_protocole_sans_augmentation": empreinte_fichier(fichier_protocole_reference),
    "modeles": INFORMATIONS, "augmentation_entrainement": AUGMENTATION_ATTENDUE,
    "limite": "Augmentation choisie apres observation des perturbations sur validation; epochs selectionnees sur cette validation; test deja consulte; comparaison exploratoire sur trois graines; niveaux hors plage choisis experimentalement.",
}
ID_PROTOCOLE = hashlib.sha256(json.dumps(PROTOCOLE, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()
SORTIE.mkdir(parents=True, exist_ok=True)
(SORTIE / "predictions").mkdir(exist_ok=True)
fichier_protocole = SORTIE / "protocole.json"
if fichier_protocole.exists():
    exiger(json.loads(fichier_protocole.read_text(encoding="utf-8")) == PROTOCOLE,
           "Le protocole ou les fichiers ont changé. Conserver le dossier existant et choisir un nouveau nom SORTIE pour une nouvelle analyse.")
else:
    ecrire_json(PROTOCOLE, fichier_protocole)
figure_exemples.savefig(SORTIE / "exemples_perturbations.png", dpi=160, bbox_inches="tight")
plt.close(figure_exemples)
exiger(len(MODELES) == 12, "Il faut douze modèles chargés.")
print("Référence de 42 résultats vérifiée. Douze modèles (6 sans, 6 avec augmentation) chargés et figés.")
display(table_modeles[["modele", "graine", "lr", "dropout", "epoque_retenue", "val_f1_macro"]].round(5))


## 4. Calculer les prédictions et mesurer les changements

Les colonnes `variation_*` représentent **mesure perturbée − mesure originale**, pour le même modèle et la même graine. Une valeur négative indique une dégradation. Par exemple, −0,02 en exactitude signifie une baisse de 2 points de pourcentage.

Le nombre de nouvelles erreurs et le nombre d’erreurs corrigées complètent cette comparaison. Les scores softmax enregistrés ne sont pas des probabilités cliniques calibrées.


In [ ]:
class ValidationPerturbee(Dataset):
    def __init__(self, condition):
        self.condition = condition
    def __len__(self):
        return len(val_df)
    def __getitem__(self, indice):
        x = transformer(PREPAREES[indice], MASQUES[indice], self.condition, IDENTIFIANTS[indice])
        return torch.from_numpy(x).unsqueeze(0), indice

def probabilites_pneumonie(logits):
    z = np.asarray(logits, dtype=np.float64)
    z = z - z.max(axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z[:, 1] / exp_z.sum(axis=1)

def verifier_predictions(tableau, information, condition):
    requis = {"chemin_relatif", "groupe", "classe_vraie", "prediction", "logit_NORMAL", "logit_PNEUMONIA", "score_pneumonie", "variante", "modele", "graine", "epoque", "condition", "id_protocole"}
    exiger(requis.issubset(tableau.columns), "Fichier de prédictions incomplet.")
    exiger(len(tableau) == len(val_df) and tableau["chemin_relatif"].tolist() == IDENTIFIANTS, "Ordre ou nombre d’images incompatible.")
    exiger(tableau["groupe"].astype(str).tolist() == val_df["groupe"].tolist(), "Groupes incompatibles.")
    exiger(np.array_equal(tableau["classe_vraie"].to_numpy(), REELS), "Étiquettes incompatibles.")
    for colonne, attendu in {"variante": information["variante"], "modele": information["modele"], "graine": information["graine"], "epoque": information["epoque"], "condition": condition["id"], "id_protocole": ID_PROTOCOLE}.items():
        exiger((tableau[colonne] == attendu).all(), f"Prédictions incompatibles : {colonne}.")
    logits = tableau[["logit_NORMAL", "logit_PNEUMONIA"]].to_numpy(dtype=np.float64)
    exiger(np.isfinite(logits).all(), "Scores bruts non finis.")
    exiger(np.array_equal(logits.argmax(axis=1), tableau["prediction"].to_numpy()), "Classes et scores bruts incohérents.")
    exiger(np.allclose(probabilites_pneumonie(logits), tableau["score_pneumonie"], rtol=0, atol=1e-10), "Scores softmax incohérents.")

def obtenir_predictions(information, condition):
    variante, nom, graine = information["variante"], information["modele"], information["graine"]
    destination = SORTIE / "predictions" / f"{variante}__{nom}_graine_{graine}__{condition['id']}.csv"
    if destination.exists():
        tableau = pd.read_csv(destination, dtype={"groupe": str})
        verifier_predictions(tableau, information, condition)
        return tableau, True
    modele = MODELES[(variante, nom, graine)]
    chargeur = DataLoader(ValidationPerturbee(condition), batch_size=LOT, shuffle=False, num_workers=0, drop_last=False)
    logits = np.empty((len(val_df), 2), dtype=np.float32)
    vus = np.zeros(len(val_df), dtype=bool)
    modele.eval()
    with torch.no_grad():
        for images, indices in tqdm(chargeur, desc=f"{variante} / {nom} / {graine} / {condition['id']}", leave=False):
            scores = modele(images.to(APPAREIL))
            exiger(scores.shape == (len(indices), 2) and torch.isfinite(scores).all().item(), "Sortie du modèle incorrecte.")
            positions = indices.numpy()
            logits[positions] = scores.cpu().numpy()
            vus[positions] = True
    exiger(vus.all(), "Des images n’ont pas été évaluées.")
    tableau = val_df[["chemin_relatif", "groupe"]].copy()
    tableau["classe_vraie"] = REELS
    tableau["prediction"] = logits.argmax(axis=1)
    tableau["logit_NORMAL"], tableau["logit_PNEUMONIA"] = logits[:, 0], logits[:, 1]
    tableau["score_pneumonie"] = probabilites_pneumonie(logits)
    for cle in ["variante", "modele", "graine", "epoque"]:
        tableau[cle] = information[cle]
    tableau["condition"], tableau["id_protocole"] = condition["id"], ID_PROTOCOLE
    verifier_predictions(tableau, information, condition)
    ecrire_csv(tableau, destination)
    return tableau, False

def calculer_mesures(tableau):
    y, prediction = tableau["classe_vraie"].to_numpy(), tableau["prediction"].to_numpy()
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    return {
        "exactitude": float(accuracy_score(y, prediction)),
        "precision_pneumonie": float(precision_score(y, prediction, zero_division=0)),
        "rappel_pneumonie": float(recall_score(y, prediction, zero_division=0)),
        "specificite_NORMAL": float(tn / (tn + fp)),
        "f1_pneumonie": float(f1_score(y, prediction, zero_division=0)),
        "f1_macro": float(f1_score(y, prediction, labels=[0, 1], average="macro", zero_division=0)),
        "auc_roc": float(roc_auc_score(y, tableau["score_pneumonie"])),
        "vrais_negatifs": int(tn), "faux_positifs": int(fp),
        "faux_negatifs": int(fn), "vrais_positifs": int(tp),
    }

def comparer_predictions(reference, perturbee):
    p0, p1 = reference["prediction"].to_numpy(), perturbee["prediction"].to_numpy()
    bon0, bon1 = p0 == REELS, p1 == REELS
    nouvelles = int(np.sum(bon0 & ~bon1))
    corrigees = int(np.sum(~bon0 & bon1))
    return {
        "predictions_changees": int(np.sum(p0 != p1)),
        "taux_predictions_changees": float(np.mean(p0 != p1)),
        "nouvelles_erreurs": nouvelles, "erreurs_corrigees": corrigees,
        "fraction_correctes_devenues_incorrectes": float(nouvelles / bon0.sum()) if bon0.any() else None,
    }
print("Fonctions prêtes. La cellule suivante lance les évaluations.")


## 5. Évaluer les douze modèles

**Étape 1 — modèles sans augmentation, 30 passages.** Chacun est d’abord réévalué sur l’originale : ses mesures doivent retrouver exactement celles du notebook 08 v2. Les quatre conditions hors plage sont ensuite évaluées. Les sept conditions du notebook 08 v2 ne sont pas recalculées : leurs résultats sont réutilisés.

**Étape 2 — modèles avec augmentation, 66 passages.** L’originale doit retrouver les scores du notebook 09 v2, puis les dix perturbations sont évaluées.

Aucun entraînement ni ajustement n’est effectué. Les résultats sauvegardés sont réutilisés si le notebook est relancé. Prévoir plusieurs dizaines de minutes selon le PC. Si une vérification échoue, conserver le message pour examiner la différence avant de continuer.


In [ ]:
METRIQUES = ["f1_macro", "exactitude", "precision_pneumonie", "rappel_pneumonie", "specificite_NORMAL", "f1_pneumonie", "auc_roc"]
METRIQUES_CONTROLE = ["f1_macro", "exactitude", "rappel_pneumonie", "specificite_NORMAL"]

def evaluer(variante, conditions, originaux_attendus):
    exiger(conditions[0]["id"] == "originale", "L’originale doit être évaluée en premier.")
    infos = [i for i in INFORMATIONS if i["variante"] == variante]
    exiger(len(infos) == 6, f"Six modèles attendus pour la variante {variante}.")
    references, mesures_originales, lignes = {}, {}, []
    total = len(conditions) * len(infos)
    for condition in conditions:
        for information in infos:
            cle = (information["modele"], information["graine"])
            debut = time.perf_counter()
            predictions, reutilise = obtenir_predictions(information, condition)
            mesures = calculer_mesures(predictions)
            if condition["id"] == "originale":
                for metrique, attendu in originaux_attendus[cle].items():
                    exiger(abs(mesures[metrique] - attendu) < 1e-10,
                           f"[{variante}] Résultat original différent pour {cle}, {metrique} : {mesures[metrique]:.8f} au lieu de {attendu:.8f}. Vérifier les sauvegardes et la préparation avant de continuer.")
                references[cle] = predictions
                mesures_originales[cle] = mesures
            original = mesures_originales[cle]
            ligne = {
                "modele": cle[0], "graine": cle[1], "epoque": information["epoque"],
                "condition": condition["id"], "libelle": condition["libelle"], "n_images": len(predictions),
                **mesures, **comparer_predictions(references[cle], predictions),
                **{f"original_{m}": original[m] for m in METRIQUES},
                **{f"variation_{m}": mesures[m] - original[m] for m in METRIQUES},
            }
            lignes.append(ligne)
            etat = "réutilisé" if reutilise else f"calculé en {(time.perf_counter() - debut):.1f} s"
            print(f"[{variante}] {len(lignes):02d}/{total} | {cle[0]} | graine {cle[1]} | {condition['id']} | F1 {mesures['f1_macro']:.5f} | variation {ligne['variation_f1_macro']:+.5f} | {etat}")
        if condition["id"] == "originale":
            print(f"[{variante}] Les six résultats originaux correspondent à la référence. Début des perturbations.")
    return pd.DataFrame(lignes)

# ---------- Étape 1 : sans augmentation, conditions hors plage ----------
originaux_sans = {}
for (nom, graine), groupe in RESULTATS_SANS_REF[RESULTATS_SANS_REF["condition"] == "originale"].groupby(["modele", "graine"]):
    originaux_sans[(nom, int(graine))] = {m: float(groupe[m].iloc[0]) for m in METRIQUES_CONTROLE}
nouveau_sans = evaluer("sans", [CONDITIONS[0]] + CONDITIONS_HORS_PLAGE, originaux_sans)
nouveau_sans = nouveau_sans[nouveau_sans["condition"] != "originale"]
RESULTATS_SANS = pd.concat([RESULTATS_SANS_REF, nouveau_sans[RESULTATS_SANS_REF.columns]], ignore_index=True)
RESULTATS_SANS = verifier_bilan(RESULTATS_SANS, CONDITIONS)
ecrire_csv(RESULTATS_SANS, SORTIE / "resultats_robustesse_sans_augmentation_complet.csv")
print(f"Sans augmentation : {len(RESULTATS_SANS)} résultats (42 repris du notebook 08 v2 + {len(nouveau_sans)} nouveaux).")

# ---------- Étape 2 : avec augmentation, toutes les conditions ----------
originaux_avec = {}
for ligne_modele in table_modeles.to_dict("records"):
    originaux_avec[(ligne_modele["modele"], int(ligne_modele["graine"]))] = {m: float(ligne_modele[f"val_{m}"]) for m in METRIQUES_CONTROLE}
bilan = verifier_bilan(evaluer("avec", CONDITIONS, originaux_avec), CONDITIONS)
ecrire_csv(bilan, SORTIE / "resultats_robustesse_augmentation.csv")
print(f"Évaluation terminée : {len(bilan)} résultats avec augmentation, aucun entraînement effectué.")


## 6. Comparer avec et sans augmentation

Les comparaisons sont calculées **pour chaque graine**, avant toute moyenne. Les deux époques sélectionnées peuvent différer : on compare les résultats de la même règle de sélection (F1 macro de validation, maximum 20 époques, patience 4), pas des époques imposées identiques.

Le fichier principal contient le score de chaque condition avec et sans augmentation, le gain de score et le gain de variation. Par exemple :

- Gain de F1 de +0,010 : F1 supérieur de 0,010 avec augmentation dans cette condition.
- Gain de spécificité de +0,05 : spécificité supérieure de 5 points de pourcentage.
- Gain de variation positif : moindre dégradation relative au score original propre au modèle. Cela doit être confronté au gain de score absolu.

Le fichier `ecarts_CBAM_classique_par_condition.csv` donne, pour chaque graine et chaque condition, l’écart de F1 macro entre CBAM et classique, avec et sans augmentation. Les conditions hors plage sont signalées par la colonne `hors_plage`.

Une synthèse utilise les trois graines ; une autre exclut la graine 42, utilisée pendant la recherche des hyperparamètres. Les barres des figures sont des écarts-types entre graines, pas des intervalles de confiance. On ne regroupe pas toutes les perturbations dans un score global ni dans un classement unique.


In [ ]:
def comparer_bilans(sans, avec):
    verifier_bilan(sans, CONDITIONS)
    verifier_bilan(avec, CONDITIONS)
    mesures = METRIQUES_COMPARAISON
    colonnes = CLES_RESULTATS + ["epoque", "n_images", "faux_positifs", "faux_negatifs"] + mesures
    colonnes += [f"original_{m}" for m in mesures] + [f"variation_{m}" for m in mesures]
    comparaison = sans[colonnes].merge(avec[colonnes], on=CLES_RESULTATS, suffixes=("_sans", "_avec"), validate="one_to_one")
    exiger(len(comparaison) == len(COUPLES) * len(CONDITIONS), "Certaines évaluations n’ont pas de correspondance.")
    for mesure in mesures:
        comparaison[f"gain_{mesure}"] = comparaison[f"{mesure}_avec"] - comparaison[f"{mesure}_sans"]
        comparaison[f"gain_variation_{mesure}"] = comparaison[f"variation_{mesure}_avec"] - comparaison[f"variation_{mesure}_sans"]
    comparaison["difference_faux_positifs"] = comparaison["faux_positifs_avec"] - comparaison["faux_positifs_sans"]
    comparaison["difference_faux_negatifs"] = comparaison["faux_negatifs_avec"] - comparaison["faux_negatifs_sans"]
    comparaison["hors_plage"] = comparaison["condition"].isin(IDS_HORS_PLAGE)
    return comparaison

def resumer_comparaison(comparaison):
    colonnes = []
    for m in METRIQUES_COMPARAISON:
        colonnes += [f"{m}_sans", f"{m}_avec", f"variation_{m}_sans", f"variation_{m}_avec", f"gain_{m}", f"gain_variation_{m}"]
    colonnes += ["difference_faux_positifs", "difference_faux_negatifs"]
    synthese = comparaison.groupby(["modele", "condition"], sort=False)[colonnes].agg(["mean", "std"])
    synthese.columns = [f"{mesure}_{stat}" for mesure, stat in synthese.columns]
    synthese = synthese.reset_index()
    synthese["hors_plage"] = synthese["condition"].isin(IDS_HORS_PLAGE)
    return synthese

def ecarts_architectures(tableau, variante):
    pivot = tableau.pivot_table(index=["graine", "condition"], columns="modele", values="f1_macro")
    exiger(set(pivot.columns) == set(ARCHITECTURES) and not pivot.isna().any().any(), "Écarts entre architectures incomplets.")
    pivot = pivot.rename(columns={"CNN_classique": "f1_macro_classique", "CNN_CBAM": "f1_macro_CBAM"})
    pivot.columns.name = None
    pivot["ecart_CBAM_moins_classique"] = pivot["f1_macro_CBAM"] - pivot["f1_macro_classique"]
    pivot = pivot.reset_index()
    pivot.insert(0, "variante", variante)
    pivot["hors_plage"] = pivot["condition"].isin(IDS_HORS_PLAGE)
    return pivot

comparaison = comparer_bilans(RESULTATS_SANS, bilan)
synthese = resumer_comparaison(comparaison)
ecrire_csv(comparaison, SORTIE / "comparaison_robustesse_augmentation.csv")
ecrire_csv(synthese, SORTIE / "synthese_comparaison_robustesse.csv")
ecrire_csv(resumer_comparaison(comparaison[comparaison["graine"] != 42]), SORTIE / "synthese_comparaison_nouvelles_graines.csv")

ecarts = pd.concat([ecarts_architectures(RESULTATS_SANS, "sans_augmentation"), ecarts_architectures(bilan, "avec_augmentation")], ignore_index=True)
ecrire_csv(ecarts, SORTIE / "ecarts_CBAM_classique_par_condition.csv")
resume_ecarts = ecarts.groupby(["variante", "condition"], sort=False)["ecart_CBAM_moins_classique"].agg(["mean", "std", "min", "max"]).reset_index()
resume_ecarts["graines_favorables_CBAM"] = ecarts.groupby(["variante", "condition"], sort=False)["ecart_CBAM_moins_classique"].apply(lambda s: int((s > 0).sum())).to_numpy()
rang = {c["id"]: i for i, c in enumerate(CONDITIONS)}
resume_ecarts = resume_ecarts.sort_values(["variante", "condition"], key=lambda col: col.map(rang) if col.name == "condition" else col.map({"sans_augmentation": 0, "avec_augmentation": 1})).reset_index(drop=True)
resume_ecarts["hors_plage"] = resume_ecarts["condition"].isin(IDS_HORS_PLAGE)
ecrire_csv(resume_ecarts, SORTIE / "synthese_ecarts_CBAM_classique.csv")

print("Effet de l’augmentation, par architecture et condition (moyenne sur 3 graines) :")
display(synthese[["modele", "condition", "hors_plage", "f1_macro_sans_mean", "f1_macro_avec_mean", "gain_f1_macro_mean", "gain_variation_f1_macro_mean", "gain_rappel_pneumonie_mean", "gain_specificite_NORMAL_mean"]].round(5))

print("Écart CBAM − classique (F1 macro), avec et sans augmentation :")
display(resume_ecarts.round(5))

print("Détail de la luminosité × 0,90, condition la plus défavorable à CBAM avant augmentation :")
display(comparaison[comparaison["condition"] == "luminosite_090"][["modele", "graine", "f1_macro_sans", "f1_macro_avec", "gain_f1_macro", "specificite_NORMAL_sans", "specificite_NORMAL_avec", "faux_positifs_sans", "faux_positifs_avec"]].round(5))

ordre = [c["id"] for c in CONDITIONS]
etiquettes = ["Originale", "Lumière\n× 0,90", "Lumière\n× 1,10", "Contraste\n× 0,90", "Contraste\n× 1,10", "Bruit\nσ 0,01", "Bruit\nσ 0,03",
              "Lumière\n× 0,80", "Lumière\n× 1,20", "Contraste\n× 0,80", "Contraste\n× 1,20"]
exiger(len(etiquettes) == len(ordre), "Étiquettes de figure incomplètes.")
separation = len(CONDITIONS_REFERENCE) - 0.5
fig, axes = plt.subplots(1, 3, figsize=(24, 6))
for axe, mesure, titre, unite in zip(axes[:2],
        ["gain_f1_macro", "gain_variation_f1_macro"],
        ["Gain de F1 macro dans chaque condition", "Gain de variation du F1 macro"],
        ["F1 avec augmentation − F1 sans augmentation", "Variation avec augmentation − variation sans augmentation"]):
    for nom, decalage, couleur, label in [("CNN_classique", -0.08, "#2563eb", "CNN classique"), ("CNN_CBAM", 0.08, "#c66b14", "CNN avec CBAM")]:
        tableau = synthese[synthese["modele"] == nom].set_index("condition").loc[ordre]
        axe.errorbar(np.arange(len(ordre))+decalage, tableau[f"{mesure}_mean"], yerr=tableau[f"{mesure}_std"], fmt="o", capsize=3, color=couleur, label=label)
    axe.set_title(titre, fontsize=12)
    axe.set_ylabel(unite)
for nom_variante, decalage, couleur, label in [("sans_augmentation", -0.08, "#6b7280", "Sans augmentation"), ("avec_augmentation", 0.08, "#15803d", "Avec augmentation")]:
    tableau = resume_ecarts[resume_ecarts["variante"] == nom_variante].set_index("condition").loc[ordre]
    axes[2].errorbar(np.arange(len(ordre))+decalage, tableau["mean"], yerr=tableau["std"], fmt="o", capsize=3, color=couleur, label=label)
axes[2].set_title("Écart de F1 macro : CBAM − classique", fontsize=12)
axes[2].set_ylabel("Positif : CBAM meilleur")
for axe in axes:
    axe.axhline(0, color="gray", linewidth=1)
    axe.axvline(separation, color="gray", linewidth=1, linestyle="--")
    axe.set_xticks(np.arange(len(ordre)), etiquettes, fontsize=8)
    axe.grid(axis="y", alpha=0.2)
    axe.legend(fontsize=9)
    axe.text(separation + 0.1, 1.0, "hors plage →", transform=axe.get_xaxis_transform(), fontsize=8, va="top", color="gray")
fig.suptitle("Effet de l’augmentation — moyenne et écart-type sur trois graines (à droite du trait : conditions hors plage d’augmentation)")
fig.tight_layout()
fig.savefig(SORTIE / "comparaison_robustesse_augmentation.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)
ecrire_json({"id_protocole": ID_PROTOCOLE, "nouveaux_passages_sans": int(len(nouveau_sans) + 6), "nouveaux_passages_avec": len(bilan),
             "comparaisons_appariees": len(comparaison), "effectif_par_passage": len(val_df)}, SORTIE / "termine.json")
print("Fichier principal à transmettre :", SORTIE / "comparaison_robustesse_augmentation.csv")
print("Écarts entre architectures :", SORTIE / "ecarts_CBAM_classique_par_condition.csv")
print("Figure complémentaire :", SORTIE / "comparaison_robustesse_augmentation.png")


## Comment interpréter cette comparaison

- Lire le gain dans les conditions perturbées et sur l’original. Ne pas déclarer une amélioration uniquement parce qu’un modèle initialement moins performant perd moins de points.
- Examiner le rappel et la spécificité séparément, ainsi que les faux positifs et faux négatifs. Un compromis entre classes peut être caché par le F1 macro.
- Conserver tous les résultats, toutes les graines et toutes les conditions. Une moyenne de trois graines ne garantit pas que chaque entraînement progresse.
- **Conditions hors plage** : si le gain n’apparaît que sur les niveaux 0,90 et 1,10 et disparaît à 0,80 et 1,20, l’augmentation a surtout appris au modèle à tolérer les variations vues à l’entraînement. Si le gain persiste hors plage, l’effet est plus général. Dans les deux cas, rapporter les deux groupes séparément.
- **Question du mémoire** : lire `synthese_ecarts_CBAM_classique.csv`. La colonne `graines_favorables_CBAM` indique sur combien des trois graines le CBAM bat le classique dans chaque condition.
- Une amélioration éventuelle répond à ces perturbations précises sur cette validation. Elle ne démontre pas une résistance à d’autres appareils, d’autres centres ou toutes les dégradations d’image.
- Les nouvelles expériences ont été motivées par la première évaluation de robustesse. Cette chronologie doit apparaître dans le mémoire ; ces mesures restent exploratoires.

Les modèles ont été entraînés avec des changements de luminosité et de contraste. Le bruit fait partie des conditions d’évaluation, mais n’a pas été ajouté pendant leur entraînement. Une seule réalisation de bruit par image est conservée, identique au notebook 08 v2. À 1,20 de luminosité, davantage de pixels clairs sont écrêtés à 1 : cette saturation fait partie de la condition. La variabilité due à d’autres réalisations du bruit n’est pas estimée.

Les définitions de préparation, de perturbation, de prédiction et de métriques sont reprises du notebook 08. Le notebook vérifie la cohérence des références et ne modifie ni les poids des modèles ni le seuil de classification.

**Vérifications avant livraison (v2) :** syntaxe de toutes les cellules ; fonctions de bilan, d’appariement et d’écarts testées sur le fichier réel `resultats_robustesse.csv` du notebook 08 v2 ; transformations hors plage testées sur des images synthétiques. Les radiographies et les sauvegardes doivent être évaluées sur votre ordinateur ; les résultats de robustesse après augmentation ne sont pas préremplis.
